# Step 6: Training 2-Stage Random Forest

Model 2-stage mengikuti arsitektur **C23 paper** (Lin et al., 2023):

| Stage | Input | Output | Model |
|-------|-------|--------|-------|
| Stage 1 | 98 fitur | Suit (C/D/H/S/N) | RandomForest |
| Stage 2 | 98 fitur | Kategori (partscore/game/small_slam/grand_slam) | RandomForest |

Level final = konversi kategori+suit ke level minimum yang valid.

**Evaluasi:** 5-fold cross-validation × 10 kali (C23 paper Section 5).

**Prasyarat:** Jalankan `05_dataset.ipynb` terlebih dahulu.

In [ ]:
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

DATA_PROCESSED = Path(ROOT) / "data" / "processed"
RESULTS        = Path(ROOT) / "results"

DATASET_CSV  = DATA_PROCESSED / "bridge_dataset.csv"
SPLIT_PATH   = RESULTS / "metrics" / "dataset_split.pkl"
MODEL_PATH   = RESULTS / "metrics" / "rf_model.pkl"

print(f"Root proyek : {ROOT}")
print("Setup selesai.")

## 6.1 Load Dataset Split

In [ ]:
from features import FEATURE_COLS
from model import prepare_features
from sklearn.model_selection import train_test_split

if SPLIT_PATH.exists():
    split_data = joblib.load(SPLIT_PATH)
    X_train      = split_data["X_train"]
    X_test       = split_data["X_test"]
    y_suit_train = split_data["y_suit_train"]
    y_suit_test  = split_data["y_suit_test"]
    y_cat_train  = split_data["y_cat_train"]
    y_cat_test   = split_data["y_cat_test"]
    print(f"Split data dimuat dari: {SPLIT_PATH}")
else:
    # Rebuild split dari dataset CSV
    print(f"Split file tidak ditemukan, rebuild dari {DATASET_CSV}")
    df = pd.read_csv(DATASET_CSV)
    df = df.dropna(subset=["best_contract_strain", "best_contract_category"])
    available_feat = [c for c in FEATURE_COLS if c in df.columns]
    X      = prepare_features(df, available_feat)
    y_suit = df["best_contract_strain"]
    y_cat  = df["best_contract_category"]
    X_train, X_test, y_suit_train, y_suit_test = train_test_split(
        X, y_suit, test_size=0.2, stratify=y_suit, random_state=42
    )
    _, _, y_cat_train, y_cat_test = train_test_split(
        X, y_cat, test_size=0.2, stratify=y_suit, random_state=42
    )

print(f"Train : {X_train.shape[0]} sampel")
print(f"Test  : {X_test.shape[0]} sampel")
print(f"Fitur : {X_train.shape[1]}")

## 6.2 Cross-Validation (Opsional)

5-fold × 10 repeats sesuai C23 paper Section 5.

> **Waktu estimasi:** 10-20 menit untuk dataset besar.
> Set `RUN_CV = True` untuk menjalankan.

In [ ]:
from model import cross_validate

RUN_CV = False  # Ubah ke True untuk menjalankan CV

if RUN_CV:
    print("Menjalankan 5-fold CV x 10 repeats...")
    cv_results = cross_validate(X_train, y_suit_train, y_cat_train)
    print(f"\nHasil Cross-Validation:")
    print(f"  Stage 1 (Suit)     F1: {cv_results['suit_f1_mean']:.4f} ± {cv_results['suit_f1_std']:.4f}")
    print(f"  Stage 2 (Category) F1: {cv_results['category_f1_mean']:.4f} ± {cv_results['category_f1_std']:.4f}")

    CV_PATH = RESULTS / "metrics" / "cv_results.pkl"
    joblib.dump(cv_results, CV_PATH)
    print(f"CV results disimpan ke: {CV_PATH}")
else:
    print("CV dilewati. Set RUN_CV = True untuk menjalankan cross-validation.")

## 6.3 Training Model Utama

In [ ]:
from model import train, save_model

print("Training 2-Stage Random Forest...")
model = train(X_train, y_suit_train, y_cat_train)

save_model(model, MODEL_PATH)
print(f"Model disimpan ke: {MODEL_PATH}")

## 6.4 Sanity Check: Prediksi di Training Set

In [ ]:
pred_train = model.predict(X_train.head(10))
actual = pd.DataFrame({
    "suit_true":  y_suit_train.head(10).values,
    "cat_true":   y_cat_train.head(10).values,
})
comparison = pd.concat([actual.reset_index(drop=True), pred_train.reset_index(drop=True)], axis=1)
comparison["suit_ok"] = comparison["suit_true"] == comparison["pred_suit"]
comparison["cat_ok"]  = comparison["cat_true"]  == comparison["pred_category"]
print("Preview 10 prediksi pertama (training set):")
comparison

## 6.5 Preview Feature Importance

In [ ]:
imp_suit, imp_cat = model.feature_importance()

print("Top 10 Fitur Terpenting — Stage 1 (Suit):")
print(imp_suit.head(10).to_string())
print()
print("Top 10 Fitur Terpenting — Stage 2 (Category):")
print(imp_cat.head(10).to_string())

---
## Output

File yang dihasilkan:
- `results/metrics/rf_model.pkl` — model TwoStageRF tersimpan
- `results/metrics/cv_results.pkl` — hasil CV (jika RUN_CV=True)

**Langkah berikutnya:** Buka `07_evaluasi.ipynb` untuk evaluasi model.